# Baseline — najprostsze podejście (wersja 0)

Najnaiwniejszy sposób: **wszystkie akta w jeden prompt** + polecenie „jesteś prawnikiem, napisz apelację". Bez planu, bez podziału na kroki, bez selekcji dokumentów.

Ten notebook: generuje apelację baseline i **ewaluuje** ją (pokrycie + halucynacje), żeby było co porównać z agentem liniowym (`linear_walkthrough.ipynb`) i planerem (`planner_walkthrough.ipynb`).

> Odpowiednik `baseline/main.py`, ale interaktywnie.

## 0. Konfiguracja

Konfiguracja LLM (klucz, model) jest w pliku `.env`. **Baseline wymaga modelu
z dużym oknem kontekstu** (prompt ~19k tokenów) — użyj OpenAI; lokalna Ollama
domyślnie obetnie kontekst do 4096.

In [1]:
import os

# Wczytaj konfigurację LLM z pliku .env (klucz nie jest wpisywany w komórce).
from dotenv import load_dotenv

load_dotenv()

print("model:", os.environ.get("LLM_MODEL"))
print("klucz z .env wczytany:", bool(os.environ.get("LLM_API_KEY")))

model: gpt-5.4-mini
klucz z .env wczytany: True


## 1. Wczytanie akt i rozmiar promptu

Baseline wrzuca **całość** w jeden prompt. Policzmy, ile to tokenów — to sedno problemu tego podejścia.

In [2]:
from src.loader import load_all
from src.tokens import count_tokens
from baseline.main import build_context
from baseline.prompts import SYSTEM_PROMPT

docs = load_all()
context = build_context(docs)
prompt_tokens = count_tokens(SYSTEM_PROMPT + "\n\n" + context)

print(f"Dokumentów: {len(docs)}")
print(f"Znaków w kontekście: {len(context):,}")
print(f"Tokenów w promptcie (wejście): ~{prompt_tokens:,}")

Dokumentów: 16
Znaków w kontekście: 52,159
Tokenów w promptcie (wejście): ~19,341


## 2. Generowanie apelacji (jeden prompt)

Jedno wywołanie LLM — całe akta wchodzą, wychodzi gotowa apelacja.

In [11]:
from baseline.main import generate_appeal

appeal = generate_appeal(docs).tekst
print(f"Apelacja: {len(appeal):,} znaków\n")
print(appeal[:1000], "...")

Apelacja: 7,911 znaków

Sąd Apelacyjny w Krakowie
za pośrednictwem
Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie
II Wydział Karny

Sygn. akt II K 25/25

OSKARŻONY: Daniel Dzik
obrońca: radca prawny Lidia Lis

APELACJA
od wyroku Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie
z dnia 14 marca 2025 r., sygn. akt II K 25/25

Działając w imieniu oskarżonego Daniela Dzika, na podstawie art. 425 § 1 i 2 k.p.k., art. 444 k.p.k. oraz art. 427 § 1 i 2 k.p.k., zaskarżam powyższy wyrok w całości, tj. w zakresie punktów 1–9.

Zaskarżonemu wyrokowi zarzucam:

I. na podstawie art. 438 pkt 2 k.p.k. – obrazę przepisów postępowania, która mogła mieć wpływ na treść orzeczenia, tj.:
1) art. 7 k.p.k. w zw. z art. 410 k.p.k. poprzez dowolną, a nie swobodną ocenę materiału dowodowego i pominięcie wynikających z akt okoliczności przemawiających na korzyść oskarżonego, w szczególności:
   - wyjaśnień oskarżonego z rozprawy z dnia 27 lutego 2025 r., w których podał, że przed przyjazdem na cmentarz był trzeź

In [12]:
from src.output import save_appeal

path = save_appeal(appeal, "baseline")
print("Zapisano apelację do:", path)

Zapisano apelację do: /Users/ewasuknarowska/Projects/WomenInTech/data/output/apelacja_baseline_2026-06-06_155720.txt


## (Opcjonalnie) Wczytaj zapisaną apelację

Zamiast generować od nowa (sekcja 2 — płatne wywołanie LLM), wczytaj **ostatnią**
zapisaną apelację z `data/output/`. Potem możesz puścić same sekcje 3–4 (ewaluacja).

> Do halucynacji (sekcja 4) potrzebny jest jeszcze `docs` — odpal sekcję 1
> (`load_all`, bez kosztu). Do samego pokrycia (sekcja 3) wystarczy `appeal`.

In [3]:
from src.output import load_latest_appeal, latest_appeal_path

appeal = load_latest_appeal("baseline")
print("Wczytano apelację z:", latest_appeal_path("baseline"))
print(f"{len(appeal):,} znaków")

Wczytano apelację z: /Users/ewasuknarowska/Projects/WomenInTech/data/output/apelacja_baseline_2026-06-06_155720.txt
7,911 znaków


## 3. Pokrycie — czy porusza wymagane zagadnienia?

Sędzia-LLM ocenia względem `data/eval.json`.

In [4]:
from src.eval.coverage import evaluate, load_eval

cov = evaluate(appeal)
print(f"Pokrycie: {cov.covered}/{cov.total} = {cov.score:.0%}\n")

# Treść zagadnień bierzemy z klucza (results są w tej samej kolejności).
for issue, r in zip(load_eval(), cov.results):
    print(("✅" if r.covered else "❌"), issue[:200])
    print(f"   → {r.reasoning}\n")

Pokrycie: 6/12 = 50%

✅ 1. W zakresie czynu z art. 178a § 1 k.k. (zarzut pkt I z aktu oskarżenia) istotnym uchybieniem, jakiego dopuścił się Sąd Rejonowy, jest błąd w ustaleniach faktycznych, polegający na dowolnym (bez opar
   → Apelacja wyraźnie podnosi zarzut błędu w ustaleniach faktycznych co do czynu z art. 178a § 1 k.k., wskazując, że oskarżony miał być nietrzeźwy już o godz. 8.30 przy przyjeździe na cmentarz, mimo braku dowodu na tę okoliczność. Wprost argumentuje, że oskarżony nie przyznał się do tego faktu, a materiał dowodowy nie pozwalał na takie ustalenie.

✅ 1. W zakresie czynu z art. 178a § 1 k.k. (zarzut pkt I z aktu oskarżenia) istotnym uchybieniem, jakiego dopuścił się Sąd Rejonowy, jest błąd w ustaleniach faktycznych, polegający na dowolnym (bez opar
   → Apelacja faktycznie podnosi zarzut błędu w ustaleniach faktycznych oraz obrazy art. 7 i 410 k.p.k. w zakresie czynu z art. 178a § 1 k.k., wskazując, że brak było dowodu na nietrzeźwość oskarżonego w chwili przyjazdu 

## 4. Halucynacje — czy apelacja nie zmyśla faktów? (etapowo)

Zamiast jednym wywołaniem (`evaluate_grounding`), pokazujemy to **krok po kroku**:

1. **twierdzenia** wyciągnięte z apelacji,
2. **wybór dokumentów** do sprawdzenia każdego twierdzenia (po opisach akt),
3. **weryfikacja** — czy twierdzenie ma oparcie w wybranych dokumentach.

> Wymaga `docs` z sekcji 1 (`load_all`).

In [4]:
# Krok 1 — twierdzenia faktyczne wyciągnięte z apelacji
from src.eval.grounding import extract_claims

claims = extract_claims(appeal)
print(f"Wyciągnięto {len(claims)} twierdzeń:\n")
for i, c in enumerate(claims, 1):
    print(f"{i:2}. {c.claim}")

Wyciągnięto 45 twierdzeń:

 1. Sygnatura akt sprawy to II K 25/25.
 2. Oskarżonym w sprawie jest Daniel Dzik.
 3. Obrońcą oskarżonego jest radca prawny Lidia Lis.
 4. Wyrok Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie został wydany 14 marca 2025 r.
 5. Apelacja została wniesiona od wyroku Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie w sprawie o sygnaturze II K 25/25.
 6. Oskarżony Daniel Dzik złożył wyjaśnienia na rozprawie 27 lutego 2025 r.
 7. Daniel Dzik podał na rozprawie 27 lutego 2025 r., że przed przyjazdem na cmentarz był trzeźwy.
 8. Daniel Dzik podał na rozprawie 27 lutego 2025 r., że alkohol spożywał dopiero na cmentarzu w alejce.
 9. Świadek Karol Kot zeznał, że nie widział, kiedy oskarżony wjechał na teren cmentarza.
10. Świadek Karol Kot zeznał, że nie widział, w jaki sposób oskarżony wjechał na teren cmentarza.
11. Świadek Karol Kot zeznał, że nie widział, by oskarżony odjeżdżał.
12. Parafia Rzymskokatolicka pw. Św. Krzysztofa w Szczyglicach odpowiedziała pismem 

### Krok 2 — wybór dokumentów do każdego twierdzenia

LLM na podstawie **opisów** akt (`file_description`) wskazuje, w których plikach
sprawdzić dany fakt — bez wczytywania wszystkich akt naraz.

In [ ]:
from src.descriptions import get_descriptions
from src.eval.grounding import select_sources

# Opisy z cache: liczone raz i współdzielone z agentami (data/output/described.json).
described = get_descriptions(docs)
selections = [(claim, select_sources(claim.claim, described)) for claim in claims]

for claim, sel in selections:
    print(f"• {claim.claim}")
    print(f"  → wybrane pliki: {sel.files}\n")

### Krok 3 — weryfikacja: czy twierdzenie to halucynacja?

Każde twierdzenie sprawdzamy **tylko** w wybranych dokumentach:
`supported` (potwierdzone) / `unsupported` (brak oparcia) / `contradicted` (sprzeczne).

In [ ]:
from collections import Counter

from src.eval.grounding import verify_claim
from src.sources import prepare_input_texts

ICON = {"supported": "✅", "unsupported": "⚠️", "contradicted": "❌"}
statuses = []
for claim, sel in selections:
    sources_text = prepare_input_texts(docs, sel.files)
    v = verify_claim(claim.claim, sources_text)
    statuses.append(v.status)
    print(f"{ICON.get(v.status, '?')} [{v.status}] {claim.claim}")
    print(f"   → {v.reasoning}\n")

counts = Counter(statuses)
n = len(statuses)
rate = (counts["unsupported"] + counts["contradicted"]) / n if n else 0
print(f"Podsumowanie: {dict(counts)} | wskaźnik halucynacji: {rate:.0%}")

## Podsumowanie

Baseline = **1 wywołanie LLM** z ogromnym promptem (patrz tokeny w sekcji 1). Szybki do napisania, ale: zapchany kontekst, „lost in the middle", brak śladu rozumowania i podatność na pominięcia/halucynacje.